In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import shutil
from tqdm import tqdm
from collections import Counter
import json
import random
from sklearn.metrics import accuracy_score, f1_score, classification_report

# =====================================================================
# 1. SETUP & PATH CONFIGURATION (KAGGLE UNZIPPED PATHS)
# =====================================================================

## Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Pre-extracted input paths from Kaggle's setup
BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
UNLABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'

# Output paths in the writable workspace
OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'
LABELED_DIR = '/kaggle/working/data/labeled_real'

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(UNLABELED_DIR):
    unlabeled_count = len([f for f in os.listdir(UNLABELED_DIR) if f.endswith('.jpg')])
    print(f"Verified dataset. Found {unlabeled_count} images in Kaggle source input directory.")
else:
    raise FileNotFoundError(f"Could not locate image directory at {UNLABELED_DIR}")

# =====================================================================
# 2. FILTERING LABELED IMAGES & METADATA POOL
# =====================================================================

print("\nFiltering to labeled images...")
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

found = 0
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        found += 1

print(f"Copied/verified {found}/{len(all_images)} labeled images into working memory.")

## Loading unlabeled metadata
unlabeled_df = pd.read_csv(os.path.join(BACKUP_DIR, 'unlabeled_pool_24k.csv'))
unlabeled_ids = unlabeled_df['image'].tolist()

## Filter to only images that actually exist
available_ids = []
missing = []
for img_id in unlabeled_ids:
    if os.path.exists(f'{UNLABELED_DIR}/{img_id}.jpg'):
        available_ids.append(img_id)
    else:
        missing.append(img_id)

print(f"\nTotal in SSL metadata: {len(unlabeled_ids)}")
print(f"Available on disk: {len(available_ids)}")
if missing:
    print(f"Missing (not on disk): {len(missing)}")

unlabeled_sample = available_ids
print(f"\nUsing {len(unlabeled_sample)} real unlabeled images for SSL Pre-training.")
print(f"Labeled splits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# =====================================================================
# 3. SimCLR COMPONENTS DEFINITION
# =====================================================================

class SimCLRTransform:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)

class UnlabeledDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.transform = transform
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = Image.open(f'{self.image_dir}/{img_id}.jpg').convert('RGB')
        return self.transform(img)

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SimCLR(nn.Module):
    def __init__(self, encoder, projection_dim=64):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, projection_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = F.normalize(self.projector(h), dim=1)
        return h, z

def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.mm(z, z.t()) / temperature

    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)

    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ])

    log_sum_exp = torch.logsumexp(sim_matrix, dim=1)
    loss = - (pos_sim - log_sum_exp)
    return loss.mean()

print("SimCLR structural elements compiled.")

# =====================================================================
# 4. SELF-SUPERVISED LEARNING (SSL) PRE-TRAINING
# =====================================================================

encoder = SimpleEncoder()
simclr_model = SimCLR(encoder).to(device)

simclr_transform = SimCLRTransform()
unlabeled_dataset = UnlabeledDataset(unlabeled_sample, UNLABELED_DIR, simclr_transform)
unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True,
    num_workers=2
)

optimizer = torch.optim.Adam(simclr_model.parameters(), lr=0.0003)

print(f"\n{'='*40}")
print(f"Training SimCLR on {len(unlabeled_sample)} Images")
print(f"Device Configuration: {device}")
print(f"{'='*40}\n")

epochs = 20
checkpoint_dir = f'{OUTPUT_DIR}/ssl_checkpoints_real_24k'
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    simclr_model.train()
    total_loss = 0
    pbar = tqdm(unlabeled_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for view1, view2 in pbar:
        view1, view2 = view1.to(device), view2.to(device)
        _, z1 = simclr_model(view1)
        _, z2 = simclr_model(view2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(unlabeled_loader)
    print(f"Epoch {epoch+1}/{epochs}, Avg Contrastive Loss: {avg_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        ckpt_path = f'{checkpoint_dir}/ssl_real_epoch_{epoch+1}.pth'
        torch.save(encoder.state_dict(), ckpt_path)
        print(f"Checkpoint saved: {ckpt_path}")

final_path = f'{OUTPUT_DIR}/ssl_encoder_real_{len(unlabeled_sample)}.pth'
torch.save(encoder.state_dict(), final_path)
print(f"\nFinal SSL encoder weights committed to: {final_path}")

# =====================================================================
# 5. DATA AUGMENTATION & CLASSIFICATION MODULES
# =====================================================================

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, mel_multiplier=5, transform_normal=None, transform_mel=None):
        self.df = df
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True)\
                          .sample(frac=1, random_state=42).reset_index(drop=True)

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img = Image.open(f"{LABELED_DIR}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
        return img, label

train_ds = AugmentedDataset(train_df, 5, normal_transform, mel_transform)
val_ds = AugmentedDataset(val_df, 1, test_transform, test_transform)
test_ds = AugmentedDataset(test_df, 1, test_transform, test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

encoder = SimpleEncoder()
encoder.load_state_dict(torch.load(final_path, map_location=device))

for param in encoder.parameters():
    param.requires_grad = True

model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
weights_list = [weight_dict[cls] for cls in train_ds.classes]
weights = torch.tensor(weights_list, dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=weights, gamma=1.5)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# =====================================================================
# 6. FINE-TUNING & EVALUATION PIPELINE — 3-SEED ROBUSTNESS RUN
# =====================================================================
#
# CHECKPOINT-SELECTION FIX (per supervisor review):
# Previously the "best" checkpoint each epoch was the one with the highest
# plain validation ACCURACY, even though the model is trained with a
# melanoma-sensitive focal loss. That is an internal inconsistency: a
# paper arguing overall accuracy is unsafe for this problem was itself
# using accuracy to decide which checkpoint to keep.
#
# Fix: select checkpoints by validation MACRO-F1 instead. Macro-F1
# averages the per-class F1 score across BKL/MEL/NV equally, so it
# directly penalizes majority-class collapse (like plain accuracy does)
# while also penalizing the opposite failure mode -- a checkpoint that
# inflates MEL recall by over-predicting MEL and destroying BKL/NV
# precision. Plain MEL recall alone was considered but rejected as the
# checkpoint criterion because it can be trivially maximized by a
# degenerate all-MEL classifier; macro-F1 does not reward that.
# Validation accuracy is still computed and logged every epoch for
# comparison/reporting, it is simply no longer the selection criterion.

def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds

seeds = [42, 123, 2024]
all_results = {}

for seed in seeds:
    print(f"\n{'#'*40}")
    print(f"SEED {seed}")
    print(f"{'#'*40}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh datasets each seed (own shuffle order, independent of other seeds)
    train_ds = AugmentedDataset(train_df, 5, normal_transform, mel_transform)
    train_ds.expanded_df = train_ds.expanded_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    val_ds  = AugmentedDataset(val_df, 1, test_transform, test_transform)
    test_ds = AugmentedDataset(test_df, 1, test_transform, test_transform)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=16)
    test_loader  = DataLoader(test_ds, batch_size=16)

    # Fresh encoder + classifier each seed, reloading the SAME SSL-pretrained weights
    # (no need to redo the 20-epoch SSL pretraining -- that part stays fixed)
    encoder = SimpleEncoder()
    encoder.load_state_dict(torch.load(final_path, map_location=device))
    for param in encoder.parameters():
        param.requires_grad = True
    model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

    weights_list = [weight_dict[cls] for cls in train_ds.classes]
    weights = torch.tensor(weights_list, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=weights, gamma=1.5)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    # Train with early stopping (patience now tracks macro-F1, not accuracy)
    epochs = 30
    best_val_macro_f1 = -1.0
    patience = 7
    epochs_no_improve = 0
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}
    seed_ckpt = f'/kaggle/working/best_seed_{seed}.pth'
    best_epoch = -1

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader)
        val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_macro_f1)
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, "
              f"Val Acc={val_acc:.3f}, Val Macro-F1={val_macro_f1:.3f}")

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            torch.save(model.state_dict(), seed_ckpt)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    print(f"Best checkpoint: epoch {best_epoch} (Val Macro-F1={best_val_macro_f1:.3f})")

    # Evaluate this seed's best checkpoint
    model.load_state_dict(torch.load(seed_ckpt, map_location=device))
    test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)

    report = classification_report(true_labels, pred_labels,
                                    target_names=train_ds.classes, output_dict=True)
    mel_recall = report['MEL']['recall']

    print(f"\n>>> Seed {seed} RESULTS: Test Acc={test_acc:.3f}, "
          f"Test Macro-F1={test_macro_f1:.3f}, MEL Recall={mel_recall:.1%} <<<")
    print(f"Prediction distribution: {Counter(pred_labels)}")

    # Save this seed's predicted probabilities too (useful for ROC/PR later)
    model.eval()
    all_probs, prob_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            probs = torch.softmax(model(images.to(device)), dim=1).cpu().numpy()
            all_probs.append(probs)
            prob_labels.extend(labels.numpy())
    all_probs = np.concatenate(all_probs)
    df_probs = pd.DataFrame(all_probs, columns=train_ds.classes)
    df_probs["true_label"] = [train_ds.classes[i] for i in prob_labels]
    df_probs.to_csv(f'{OUTPUT_DIR}/test_probs_seed{seed}.csv', index=False)

    all_results[seed] = {
        "test_accuracy": test_acc,
        "test_macro_f1": test_macro_f1,
        "mel_recall": mel_recall,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "report": report,
        "history": history,
        "checkpoint_selection_metric": "val_macro_f1",
    }
    torch.save(model.state_dict(), f'{OUTPUT_DIR}/ssl_rebalance_seed{seed}_finetuned.pth')

# =====================================================================
# 3-SEED SUMMARY (same reporting format as your baseline Table 4.1)
# =====================================================================
accs = [all_results[s]["test_accuracy"] for s in seeds]
f1s = [all_results[s]["test_macro_f1"] for s in seeds]
recalls = [all_results[s]["mel_recall"] for s in seeds]

print("\n" + "="*40)
print("3-SEED SUMMARY -- SSL + Rebalance (checkpoint selected by val macro-F1)")
print("="*40)
for s in seeds:
    print(f"Seed {s}: Test Acc={all_results[s]['test_accuracy']:.3f}, "
          f"Test Macro-F1={all_results[s]['test_macro_f1']:.3f}, "
          f"MEL Recall={all_results[s]['mel_recall']:.1%}, "
          f"Best epoch={all_results[s]['best_epoch']}")
print(f"\nMean Test Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Mean Test Macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
print(f"Mean MEL Recall:    {np.mean(recalls):.1%} +/- {np.std(recalls):.1%}")

with open(f'{OUTPUT_DIR}/ssl_rebalance_3seed_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved full 3-seed results to {OUTPUT_DIR}/ssl_rebalance_3seed_results.json")


Device: cuda
Verified dataset. Found 25331 images in Kaggle source input directory.

Filtering to labeled images...
Copied/verified 691/691 labeled images into working memory.

Total in SSL metadata: 24000
Available on disk: 24000

Using 24000 real unlabeled images for SSL Pre-training.
Labeled splits: Train=483, Val=104, Test=104
SimCLR structural elements compiled.

Training SimCLR on 24000 Images
Device Configuration: cuda



Epoch 1/20: 100%|██████████| 375/375 [08:29<00:00,  1.36s/it, loss=3.5833]


Epoch 1/20, Avg Contrastive Loss: 3.7138


Epoch 2/20: 100%|██████████| 375/375 [06:38<00:00,  1.06s/it, loss=3.3900]


Epoch 2/20, Avg Contrastive Loss: 3.4840


...

Epoch 18/20, Avg Contrastive Loss: 3.1801


Epoch 19/20: 100%|██████████| 375/375 [06:40<00:00,  1.07s/it, loss=3.1603]


Epoch 19/20, Avg Contrastive Loss: 3.1717


Epoch 20/20: 100%|██████████| 375/375 [06:41<00:00,  1.07s/it, loss=3.1309]


Epoch 20/20, Avg Contrastive Loss: 3.1667
Checkpoint saved: /kaggle/working/Thesis_Outputs/ssl_checkpoints_real_24k/ssl_real_epoch_20.pth

Final SSL encoder weights committed to: /kaggle/working/Thesis_Outputs/ssl_encoder_real_24000.pth

########################################
SEED 42
########################################
Epoch 1: Train Acc=0.278, Val Acc=0.183, Val Macro-F1=0.138
Epoch 2: Train Acc=0.387, Val Acc=0.462, Val Macro-F1=0.278
Epoch 3: Train Acc=0.555, Val Acc=0.538, Val Macro-F1=0.275
Epoch 4: Train Acc=0.600, Val Acc=0.606, Val Macro-F1=0.376
Epoch 5: Train Acc=0.650, Val Acc=0.654, Val Macro-F1=0.432
Epoch 6: Train Acc=0.644, Val Acc=0.673, Val Macro-F1=0.464
Epoch 7: Train Acc=0.669, Val Acc=0.635, Val Macro-F1=0.437
Epoch 8: Train Acc=0.658, Val Acc=0.635, Val Macro-F1=0.449
Epoch 9: Train Acc=0.682, Val Acc=0.654, Val Macro-F1=0.469
Epoch 10: Train Acc=0.708, Val Acc=0.625, Val Macro-F1=0.457
Epoch 11: Train Acc=0.695, Val Acc=0.654, Val Macro-F1=0.468
Epoch 12: 

In [3]:
# Save outputs to a persistent dataset (requires kagglehub or manual download)
# Option A: Download to your local machine
import zipfile
import os
shutil.make_archive('/kaggle/working/Thesis_Outputs', 'zip', '/kaggle/working/Thesis_Outputs')
# Then manually download the zip from the Kaggle UI.

# Option B: Upload to a custom Kaggle dataset using the Kaggle API
# (This requires API keys – but for now, just download the zip manually).

'/kaggle/working/Thesis_Outputs.zip'

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os

# --- Paths (update these to match your setup) ---
BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR = '/kaggle/working/data/labeled_real'  # or where images are
CKPT_DIR = '/kaggle/working/Thesis_Outputs'  # or local folder
OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Model defs (copy from your notebook) ---
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

# --- Test Data ---
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
classes = sorted(test_df['label'].unique())

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class SimpleTestDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df
        self.image_dir = image_dir
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.image_dir}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

test_ds = SimpleTestDataset(test_df, LABELED_DIR, test_transform)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

# --- Inference loop for each seed ---
seeds = [42, 123, 2024]
for seed in seeds:
    ckpt_path = f'{CKPT_DIR}/ssl_rebalance_seed{seed}_finetuned.pth'
    if not os.path.exists(ckpt_path):
        print(f"Checkpoint not found: {ckpt_path}")
        continue
    encoder = SimpleEncoder()
    model = SSLClassifier(encoder, len(classes)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    
    all_probs = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            probs = torch.softmax(model(images), dim=1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs)
    
    df_probs = pd.DataFrame(all_probs, columns=classes)
    df_probs['true_label'] = test_df['label'].values
    df_probs.to_csv(f'{OUTPUT_DIR}/test_probs_seed{seed}.csv', index=False)
    print(f"Saved {OUTPUT_DIR}/test_probs_seed{seed}.csv")

Saved /kaggle/working/Thesis_Outputs/test_probs_seed42.csv
Saved /kaggle/working/Thesis_Outputs/test_probs_seed123.csv
Saved /kaggle/working/Thesis_Outputs/test_probs_seed2024.csv
